# Part 3 — Finding Similar Structures with a k-d Tree

In **Part 2** we compared chains *pairwise* — every chain against every other,
shown as a distance matrix. That works for a few dozen chains. But a very common
question is the opposite:

> "I have **one** structure. Which structures in a **database** are most similar
> to it?"

Checking the whole database one entry at a time (brute force) becomes expensive.
This notebook shows a faster strategy based on a **k-d tree**, through the
function `neighbour_distance_search_ckdtree()`.

## Table of Contents

1. [Setup](#setup) 
2. [The nearest-neighbour problem and the k-d tree (concept)](#kdtree) 
3. [Preparing one query chain and a reference set](#prep) 
4. [Running `neighbour_distance_search_ckdtree()`](#nndist) 
5. [Interactive visualisation with **Plotly**](#visplotly)
6. [Experimental vs Computational](#6-experimental-vs-computational)
6. [Quick reference](#quickref)


## The biological question

You have selected **one protein chain** — the *query* — and want to rank every
other chain in a set by how close its backbone geometry is to the query,
**without superposing** anything. Because BRI is rotation/translation-invariant,
"similar geometry" is simply a "small BRI distance".


## 1. Setup <a id="setup"></a>

In [ ]:
!pip install plotly
import bri
import numpy as np
import pandas as pd

import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

from bri import ProteinEntry
from bri.invariant import BRI_COLUMNS
from bri.invariant_compare import neighbour_distance_search_ckdtree

print(f"bri version: {bri.__version__}")

## 2. The searching problem <a id="kdtree"></a>

### 2.1 The problem

Each chain is represented by one **BRI vector** — a list of numbers describing
its backbone geometry. Comparing two chains then reduces to the *distance*
between their two vectors (here, the **Chebyshev** or L∞ distance: the single
largest coordinate difference).

Given one **query** vector and a **database** of *N* reference vectors, the
naive approach measures the distance to every reference and sorts the results.
That is **O(N)** comparisons — instantaneous for 20 chains, but slow when *N*
reaches thousands or millions.

#### Question: Do we have a better way to make the computation faster?

### 2.2 What is a k-d tree?

A **k-dimensional tree (k-d tree)** is a way of *indexing* points in space so
that nearest neighbours can be found **without measuring the distance to every
point**.

> A plain analogy: to find a word in a dictionary you do not read every page
> from the front. The alphabetical *ordering* lets you jump near the target and
> discard most of the book at each step. A k-d tree plays the same role for
> points in space: it splits space into regions, so whole regions can be skipped
> when they cannot possibly contain a nearer neighbour.

Conceptually, the tree splits the data along alternating axes:

```
            [split along axis 1]
              /              \
      [split axis 2]      [split axis 2]
        /       \             /       \
     (leaf)   (leaf)      (leaf)    (leaf)
```

Each **leaf** stores a few points; each **internal node** records a splitting
plane. A nearest-neighbour query drops down the tree to the region containing the
query, then backtracks checking only nearby regions — turning an *O(N)* scan into
roughly *O(log N)* work for low-dimensional data.

### 2.3 Why it matters for BRI

- **Each chain becomes one point** in a space whose axes are the BRI values.
  Similar structures cluster together; dissimilar ones sit far apart.
- `neighbour_distance_search_ckdtree()` builds a k-d tree over the reference set
  and queries it with your chain, returning the neighbours **sorted
  nearest-first**.
- **Caveat — the "curse of dimensionality":** k-d trees lose their advantage as
  the number of dimensions grows, because the splitting regions can no longer
  rule out large parts of the space. BRI vectors can be high-dimensional
  (roughly *9 × chain length* numbers per chain), so for very long chains the
  speed-up over brute force shrinks. For the small sets used here the search is
  instantaneous either way; the tree truly pays off for **large structural
  databases of short-to-medium chains**.

<!--
TODO — expand this section:
  - a figure of a 2-D k-d tree partition with a query point
  - a complexity table (brute force vs k-d tree: best / average / worst case)
  - when to prefer a BallTree (used by the RMS variant) over a cKDTree
-->

## 3. Prepare a query and a reference set <a id="prep"></a>

We reuse the **2K4P NMR ensemble** from Part 2: the same protein solved as
**20 structural models**. All 20 models share **one amino-acid sequence** but
differ slightly in 3D conformation — an ideal test bed, because it lets us rank
conformations of a *single sequence* by structural distance.

- **Query** — one model (we pick model 1). This is "the sequence we care about".
- **Reference set** — the *other* 19 models.

We first compute the BRI of every model and tag each with a synthetic `pdb_id`
(`model_1` … `model_20`) so the search results are easy to read back.

In [ ]:
nmr_entry = ProteinEntry.from_cif("2k4p")
poly_chains = [c for c in nmr_entry.chains if c.polypeptide]

frames = []
for chain in poly_chains:
    inv = chain.get_invariant(invariant_type="bri")
    inv["pdb_id"] = f"model_{chain.model_id}"
    frames.append(inv)

nmr_bri = pd.concat(frames, ignore_index=True)

n_models = nmr_bri["pdb_id"].nunique()
chain_len = int(nmr_bri["chain_length"].iloc[0])
print(f"{n_models} models x {chain_len} residues = {len(nmr_bri)} rows")

In [ ]:
QUERY_ID = "model_1"

# Query: one model's per-residue BRI.
query_bri = nmr_bri[nmr_bri["pdb_id"] == QUERY_ID].reset_index(drop=True)

# Reference set: every OTHER model.
ref_bri = nmr_bri[nmr_bri["pdb_id"] != QUERY_ID].reset_index(drop=True)

# Show the input "sequence".
query_seq = "".join(query_bri["residue_label"])
print(f"Query:      {QUERY_ID}")
print(f"Length:     {len(query_seq)} residues")
print(f"Sequence:   {query_seq}")
print(f"References: {ref_bri['pdb_id'].nunique()} other models")

## 4. Search for the nearest structures <a id="nndist"></a>

`neighbour_distance_search_ckdtree(target, df)` builds a k-d tree over `df`
(the references) and returns every reference's **Chebyshev (L∞) distance** to the
single `target` (our query), **sorted from nearest to farthest**.

We pass `seq_compare=True` to also report the **sequence difference** (`seq_diff`
= number of differing residues). Because all 20 NMR models come from **one
sequence**, expect `seq_diff = 0` everywhere — the differences you see are purely
*conformational*, not sequence-based. (For a database of *different* proteins of
equal length, `seq_diff` would vary and would let you separate "same fold,
different sequence" from "nearly identical".)

In [ ]:
result = neighbour_distance_search_ckdtree(
    target=query_bri,
    df=ref_bri,
    seq_compare=True,
)

result["rank"] = range(1, len(result) + 1)
result

### Reading the table

| Column | Meaning |
|--------|---------|
| `distance` | Chebyshev (L∞) BRI distance to the query, in Å. Smaller = more similar. |
| `pdb_id` / `model_id` | Which reference model. |
| `seq_diff` | Number of residues that differ in sequence (0 here — one sequence). |
| `rank` | 1 = nearest, 19 = farthest. |

The nearest model has the backbone geometry most like the query; the farthest is
the most divergent conformation in the ensemble.

## 5. Interactive visualisation with Plotly <a id="visplotly"></a>

Two views, both interactive — hover, zoom, and toggle traces via the legend.

### 5.1 How far is each model from the query?

A ranked bar chart of the Chebyshev distance, nearest models at the top.

In [ ]:
def show_html(fig, *args, **kwargs):
    display(HTML(pio.to_html(fig, include_plotlyjs='cdn', full_html=False)))

go.Figure.show = show_html
pio.show=show_html

bar = result.sort_values("distance", ascending=False).reset_index(drop=True)

fig_a = go.Figure(go.Bar(
    x=bar["distance"],
    y=bar["pdb_id"],
    orientation="h",
    marker=dict(
        color=bar["distance"],
        colorscale="Viridis",
        colorbar=dict(title="Chebyshev<br>distance (A)", thickness=10),
    ),
    customdata=np.stack([bar["model_id"], bar["seq_diff"]], axis=-1),
    hovertemplate=(
        "<b>%{y}</b><br>"
        "distance: %{x:.3f} A<br>"
        "model_id: %{customdata[0]}<br>"
        "seq_diff: %{customdata[1]}"
        "<extra></extra>"
    ),
))

fig_a.update_layout(
    title=f"Nearest neighbours of {QUERY_ID} - Chebyshev BRI distance",
    xaxis_title="distance to query (A)",
    height=500,
)
fig_a.update_yaxes(title_text="reference model", categoryorder="total descending")
show_html(fig_a)

### 5.2 Where along the backbone do neighbours differ?

This is the interactive counterpart of `workflow.comparison_plots`: each panel is
one BRI coordinate plotted along the residue sequence.

- The **query** is the bold red line.
- The **top-K nearest** models are thin coloured lines. Use the legend to toggle
  any of them on/off; hovering shows the residue number and value.

Change `TOP_K` to overlay more or fewer neighbours.

In [ ]:
TOP_K = 5  # number of nearest neighbours to overlay (change me)

top_k = result.sort_values("distance").head(TOP_K).reset_index(drop=True)

fig_b = make_subplots(
    rows=len(BRI_COLUMNS), cols=1, shared_xaxes=True, vertical_spacing=0.015,
    subplot_titles=BRI_COLUMNS,
)


def _curve(df_model, col):
    """Residue numbers + invariant values for one model/column, in sequence order."""
    g = df_model.sort_values("residue_id")
    return g["residue_id"].astype(int), pd.to_numeric(g[col], errors="coerce")


# Query line on every panel (legend entry shown once).
for i, col in enumerate(BRI_COLUMNS, start=1):
    x, y = _curve(query_bri, col)
    fig_b.add_trace(
        go.Scatter(
            x=x, y=y, name="query (model_1)",
            mode="lines+markers", line=dict(color="crimson", width=2.5),
            marker=dict(size=3),
            legendgroup="query", showlegend=(i == 1),
            hovertemplate="residue %{x}<br>%{y:.3f}<extra>query</extra>",
        ),
        row=i, col=1,
    )

# One legendgroup per neighbour: clicking it hides that neighbour on ALL panels.
for _, nrow in top_k.iterrows():
    label = nrow["pdb_id"]
    dist = nrow["distance"]
    model_df = nmr_bri[nmr_bri["pdb_id"] == label]
    for i, col in enumerate(BRI_COLUMNS, start=1):
        x, y = _curve(model_df, col)
        fig_b.add_trace(
            go.Scatter(
                x=x, y=y, name=f"{label} (d={dist:.3f})",
                mode="lines", line=dict(width=1),
                legendgroup=label, showlegend=(i == 1),
                hovertemplate=f"residue %{{x}}<br>%{{y:.3f}}<extra>{label}</extra>",
            ),
            row=i, col=1,
        )

fig_b.update_layout(
    title=f"BRI overlay - {QUERY_ID} vs its {TOP_K} nearest neighbours",
    height=200 * len(BRI_COLUMNS),
    legend=dict(orientation="h", yanchor="bottom", y=-0.02),
    margin=dict(l=60, r=20, t=60, b=40),
)
fig_b.update_xaxes(title_text="residue number", row=len(BRI_COLUMNS), col=1)
show_html(fig_b)

**How to explore this plot**

- **Hover** over any line to read the residue number and invariant value.
- **Click a neighbour in the legend** to hide or show it across *all nine* panels.
- **Zoom** by dragging inside a panel; the x-axis is shared, so every panel zooms
  together.
- Look for panels where the coloured lines **diverge from the red line** — those
are the BRI coordinates (and thus the backbone features) that differ most between
the query and its neighbours.

> Try increasing `TOP_K` (e.g. to 10), or change `QUERY_ID` to another model and
> re-run the cells from Section 3 to re-rank the ensemble around a different
> reference.

## 6. Experimental VS Computational <a id="visaf"></a>

How predictions can look different from the experimental structure in terms of the invariants.


In [ ]:
from bri.workflow import comparison_plots

fig_dict=comparison_plots('example_data/predictions/6xpf', reference='6xpf', offset=2)

## 7. Quick reference <a id="quickref"></a>

| You want to … | Use |
|---------------|-----|
| Rank a database against one query (Chebyshev) | `neighbour_distance_search_ckdtree(target, df)` |
| Same, with RMS distance | `neighbour_distance_search_RMSD(target, df)` |
| Also report the sequence difference | `…(target, df, seq_compare=True)` |
| All-pairs comparison (Part 2) | `group_invariant_compare(data)` |

### Key takeaways

- BRI turns "structural similarity" into a **vector distance** — no superposition.
- A k-d tree makes **query-versus-database** search efficient for large sets.
- All references must share the **same chain length**; for mixed lengths, see the
  statistical projection in Part 2.

> **Note:** All 20 NMR models share one sequence, so `seq_diff` was 0 here.
> Comparing across *different* proteins of equal length works identically —
> `seq_diff` then distinguishes "same fold, different sequence" from "nearly
> identical".